# The Apple Card Credit Limit Gap: When a Black Box Can't Explain Itself

In this notebook I look at the Apple Card credit limit controversy as a sixth, distinct mechanism behind an outcome gap.

The Tokyo Medical University notebook was a deliberate manual adjustment. The COMPAS notebook was an honest score meeting differing base rates. The Amazon notebook was a neutral feature acting as an accidental proxy. The facial recognition notebook was a representation gap from underrepresented training data. The Dutch childcare benefits notebook was an explicit protected attribute paired with a zero-tolerance policy. This one is different from all five: investigators looked for the usual levers, a gender input, a single obvious proxy, and did not find them, yet a gap still showed up, traced back to a decades-old bookkeeping convention nobody had thought to question, and a bank that often could not explain its own individual decisions.

In this notebook, I will:

- Summarize what was reported and what New York's financial regulator found
- Simulate married couples with genuinely identical joint household finances
- Introduce one real, specific mechanism regulators pointed to: how joint credit accounts get individually attributed
- Measure the credit-limit gap this produces, without gender ever being a model input
- Generate the same kind of adverse action reason code a real bank sends, and see what it leaves out
- Try the fix that worked in earlier notebooks, and check what it does and doesn't resolve

## 1. Background: What Was Reported

In November 2019, software developer David Heinemeier Hansson said publicly that Apple Card had given his wife a credit limit twenty times lower than his own, despite the couple filing joint tax returns, living in a community-property state where their assets and income were shared, and his wife having a higher individual credit score than he did. Apple co-founder Steve Wozniak reported a similar experience with his wife shortly after, a roughly ten-times gap.

Apple Card's underwriting was handled by Goldman Sachs. The New York State Department of Financial Services opened an investigation. Its 2021 report found no evidence that gender was used as a direct input to the credit model, and did not establish a violation of fair lending law. It also found that Goldman could often not produce a clear, individualized explanation for why two applicants received different outcomes, and it pointed to a long-standing credit-reporting practice as a likely contributor: joint or authorized-user accounts have traditionally had their full history counted toward whichever spouse was listed as the account's primary holder, and only a partial or discounted history counted toward the other, a decades-old convention that, for many long-married couples, skewed toward crediting the husband.

## 2. Why This Is a Different Mechanism

Every earlier notebook in this series traced a gap to a specific lever sitting inside the system, a manual deduction, a threshold, a proxy feature, a training data gap, an explicit protected-attribute input. Here, investigators looked for exactly those levers and did not find them: no gender field, no single feature that obviously stood in for it.

What they found instead was two things layered on top of each other. First, a supposedly individual piece of data, a person's own credit history, was not really individual at all, it partly reflected a bookkeeping convention set decades earlier by someone else, about whose name went on an account first. Second, and more unsettling, the bank running the algorithm often could not give a specific, case-by-case reason for its own decisions, only a set of plausible contributing factors after the fact. A gap that traces back to an inherited convention, filtered through a system nobody can fully explain, is a different problem from a gap with one bad input: there's no single lever to point to and remove, only a process to make honest and a convention to question.

## 3. A Note on the Simulation Below

This notebook does not use Apple Card's or Goldman's real model, which was never fully published. I simulate one specific, real mechanism regulators pointed to, the way joint accounts get individually attributed, to show how it can manufacture a gender-correlated credit-limit gap without gender ever appearing as a model input. I also generate the kind of adverse action reason code a real underwriting system produces, to show what a technically true explanation can still leave out.

## 4. Simulating Couples With Identical Joint Household Finances

I generate married couples who share one joint household: the same joint income and the same number of years married, since in a community-property state with joint tax filing this is realistically true of both partners. Any credit-limit gap this notebook finds from here on is not coming from a real difference in the couple's underlying finances.

In [ ]:
import random

random.seed(21)

N_COUPLES = 3000


def make_couples(n):
    """Returns a list of couple dicts with identical joint income and years married for both partners."""
    couples = []
    for _ in range(n):
        couples.append({
            "joint_income": random.uniform(60000, 220000),
            "joint_years_married": random.randint(1, 30),
        })
    return couples


couples = make_couples(N_COUPLES)

## 5. The Joint Account Attribution Rule

Decades of credit-reporting practice attributed a joint or authorized-user account's full history to whichever spouse was named the account's primary holder, and only a heavily discounted history to the other, an old bookkeeping convention neither spouse chose. Because credit accounts were more often opened in a husband's name in the decades many of today's couples have been married, primary status skews toward husbands, not because of anything about the individual applying today, but because of who a bank listed first a long time ago.

In [ ]:
def assign_primary_holder():
    """Returns which spouse is the historically designated primary account holder."""
    return "husband" if random.random() < 0.78 else "wife"


for couple in couples:
    couple["primary"] = assign_primary_holder()
    full_history = couple["joint_years_married"]
    couple["husband_individual_history"] = full_history if couple["primary"] == "husband" else full_history * 0.2
    couple["wife_individual_history"] = full_history if couple["primary"] == "wife" else full_history * 0.2

## 6. Building a Credit-Limit Score From Gender-Free Individual Inputs

The score below uses only two individual numbers: each spouse's half of the joint income, since the household income is genuinely shared, and each spouse's individually-attributed credit history length from the step above. Gender never appears in this function, only numbers the attribution rule already touched.

In [ ]:
def credit_limit_score(individual_income, individual_history_years):
    """Returns a credit-limit score built only from individual income and individual credit history."""
    score = individual_income * 0.01
    score += individual_history_years * 400
    score += random.gauss(0, 300)
    return score


for couple in couples:
    individual_income = couple["joint_income"] / 2
    couple["husband_score"] = credit_limit_score(individual_income, couple["husband_individual_history"])
    couple["wife_score"] = credit_limit_score(individual_income, couple["wife_individual_history"])

## 7. Measuring the Credit-Limit Gap

I compare the average score for husbands and wives across all three thousand couples, and count how often one partner's score comes out less than half the other's, the kind of gap that produced the tweets that started this story, despite every couple here having identical joint finances by construction.

In [ ]:
avg_husband_score = sum(c["husband_score"] for c in couples) / len(couples)
avg_wife_score = sum(c["wife_score"] for c in couples) / len(couples)

lower_wife = len([c for c in couples if c["wife_score"] < c["husband_score"] * 0.5])
lower_husband = len([c for c in couples if c["husband_score"] < c["wife_score"] * 0.5])

print("Average score, husbands:", round(avg_husband_score))
print("Average score, wives:", round(avg_wife_score))
print("Couples where the wife's score is under half the husband's:", lower_wife)
print("Couples where the husband's score is under half the wife's:", lower_husband)

## 8. Confirming Where the Gap Actually Comes From

Gender was never a variable in `credit_limit_score`. To confirm the gap above is flowing entirely through the primary/secondary attribution, I check how often the historically primary-designated partner is simply the one with the higher score.

In [ ]:
def share_primary_has_higher_score(couples):
    """Returns the fraction of couples where the historically primary-designated partner has the higher score."""
    matches = len([
        c for c in couples
        if (c["primary"] == "husband") == (c["husband_score"] > c["wife_score"])
    ])
    return matches / len(couples)


print(
    "Share of couples where the primary-designated partner has the higher score:",
    round(share_primary_has_higher_score(couples), 3),
)

## 9. Generating an Adverse Action Reason Code

US law, the Equal Credit Opportunity Act, requires lenders to give applicants a specific reason when they receive a lower credit limit than expected. I generate the reason code a real underwriting system would produce for whichever spouse in each couple scored lower, based on the input that most directly drove their score down.

In [ ]:
from collections import Counter


def adverse_action_reason(individual_history_years):
    """Returns the reason code a real system would report for a lower credit-limit score."""
    if individual_history_years < 5:
        return "Insufficient length of individual credit history"
    return "Limited individual credit history relative to file"


reasons_given = []
for couple in couples:
    if couple["wife_score"] < couple["husband_score"]:
        reasons_given.append(adverse_action_reason(couple["wife_individual_history"]))
    else:
        reasons_given.append(adverse_action_reason(couple["husband_individual_history"]))

print(Counter(reasons_given))

## 10. Why a True Reason Code Can Still Be Misleading

Every one of those reason codes is factually accurate: the lower-scoring spouse's individually-attributed credit history really is shorter. But the reason code names the symptom of a decades-old attribution rule, not its cause, and gives the affected spouse nothing they can act on or contest: their individual history is short because of how a joint account happened to be labeled long ago, not because of any credit behavior of their own. A technically true explanation that omits the actual lever behind it is its own kind of unaccountability, distinct from a wrong explanation.

## 11. Trying the Fix That Worked Before

In the Dutch childcare benefits and Amazon notebooks, removing or correcting one identifiable input closed most of the gap. I try the equivalent move here: credit both spouses with the couple's full joint account history instead of splitting it into a full share for the primary holder and a fifth of it for the other, and rebuild both scores.

In [ ]:
for couple in couples:
    full_history = couple["joint_years_married"]
    individual_income = couple["joint_income"] / 2
    couple["husband_score_fixed"] = credit_limit_score(individual_income, full_history)
    couple["wife_score_fixed"] = credit_limit_score(individual_income, full_history)

## 12. Measuring the Gap After the Fix

I rerun the same comparison as before, average score by spouse and the count of couples with a more-than-half gap between partners, now that both spouses are credited with the same underlying joint history.

In [ ]:
avg_husband_fixed = sum(c["husband_score_fixed"] for c in couples) / len(couples)
avg_wife_fixed = sum(c["wife_score_fixed"] for c in couples) / len(couples)

lower_wife_fixed = len([c for c in couples if c["wife_score_fixed"] < c["husband_score_fixed"] * 0.5])
lower_husband_fixed = len([c for c in couples if c["husband_score_fixed"] < c["wife_score_fixed"] * 0.5])

print("Average score, husbands, full joint history credited:", round(avg_husband_fixed))
print("Average score, wives, full joint history credited:", round(avg_wife_fixed))
print("Couples with a wife score under half the husband's, after the fix:", lower_wife_fixed)
print("Couples with a husband score under half the wife's, after the fix:", lower_husband_fixed)

## 13. What the Regulator Actually Concluded

New York's Department of Financial Services did not find gender coded into Goldman's model, and did not establish a fair lending violation. It also explicitly flagged that Goldman could not adequately explain many of its own individual credit decisions, and recommended the industry reconsider exactly the joint-account attribution practice this notebook simulates. Those are two separate findings, and it is worth sitting with the gap between them: "we found no evidence of the specific bias we checked for" is not the same claim as "we can explain this outcome," and a system can pass the first test while still failing the second.